In [ ]:
# Find this checkout from a notebook or a kernel launched at the workspace root.
from pathlib import Path
import os, sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents, _here / 'cic17-cic18-transfer-study']
if os.environ.get('NIDS_STUDY_ROOT'):
    _candidates.insert(0, Path(os.environ['NIDS_STUDY_ROOT']).expanduser().resolve())
REPO_ROOT = next((p for p in _candidates if (p / 'src/nids_transfer').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Launch Jupyter from this checkout or set NIDS_STUDY_ROOT.')
_src = str(REPO_ROOT / 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)
from nids_transfer.paths import study_paths
DATA_DIR, ARTIFACT_DIR = study_paths(REPO_ROOT)
print('Data:', DATA_DIR)
print('Artifacts:', ARTIFACT_DIR)


# Training coverage and cross-year transfer

**Question:** are classifier errors more common for evaluation points with weak support from 2017 training examples?

Run `binary_classification_baseline.ipynb` first. Load its frozen splits, feature preprocessing, and predictions. This notebook **does not retrain classifiers** and does not use evaluation labels/scores to fit PCA or choose coverage thresholds.

Fit PCA to continuous quantitative training features only (flags and protocol are excluded from this geometric view). Retain enough unwhitened components for 95% of training variance; project both evaluation datasets with exactly that source-fitted map. PC1–PC2 plots are illustrations, while distance uses all retained components.

Use exact KD-tree neighbor search against **all training locations**, merging identical float32 PCA coordinates. Sample queries, not the reference population. If exact search becomes too costly on another dataset, use an audited approximate index rather than silently shrinking the reference population.
Coverage artifacts go in `binary_baseline_results/coverage/`, separately from the frozen classifier artifacts.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from IPython.display import display
from nids_transfer.binary_transfer_utils import (BinaryStudy,fit_coverage,coverage_reference,
    select_coverage_queries,assess_coverage,coverage_reports,read_json,write_json)
ROOT = REPO_ROOT
OUT = ARTIFACT_DIR / 'binary_baseline_results'
if not (OUT/'baseline_complete.json').exists():
    raise FileNotFoundError('Finish the baseline notebook first.')
registry = read_json(OUT/'baseline_complete.json')
PATHS = {year:DATA_DIR/f'compatible/CIC{year}_shared.parquet' for year in ['2017','2018']}
study = BinaryStudy(PATHS,OUT,seed=registry['seed'])
COVERAGE_OUT = OUT / 'coverage'  # Choose a different subdirectory for another coverage experiment.
COVERAGE_OUT.mkdir(exist_ok=True)
study.coverage_out = COVERAGE_OUT
VARIANCE_TARGET = .95
K_NEIGHBORS = 5
CALIBRATION_ROWS = 3_000
POPULATION_QUERIES_PER_SPLIT = 15_000
MINIMUM_QUERIES_PER_LABEL = 300

# Coverage configurations are versioned separately; classifier artifacts stay frozen.
import hashlib
coverage_settings = {
    'baseline_registry_sha256':hashlib.sha256((OUT/'baseline_complete.json').read_bytes()).hexdigest(),
    'variance_target':VARIANCE_TARGET,'k_neighbors':K_NEIGHBORS,
    'calibration_rows':CALIBRATION_ROWS,'population_queries_per_split':POPULATION_QUERIES_PER_SPLIT,
    'minimum_queries_per_label':MINIMUM_QUERIES_PER_LABEL,
}
settings_file=COVERAGE_OUT/'experiment_settings.json'
if settings_file.exists() and read_json(settings_file)!=coverage_settings:
    raise ValueError('Coverage settings differ: choose another COVERAGE_OUT; classifiers need no retraining.')
write_json(settings_file,coverage_settings)


## Train-only coverage coordinates

Reuse the train-fitted log means/standard deviations from the classifier preprocessing, selecting only quantitative non-flag columns. Compute the complete training covariance in batches and diagonalize it for ordinary full PCA. Neither pooled-year PCA from the earlier exploration nor an evaluation-fitted scaler is used here.

Distances retain relative PC variances: no whitening or individual PC rescaling. This geometry is one explicit representation of support, not a universal novelty score. Protocol novelty is reported separately; identical continuous coordinates can still have different protocols/flags.

In [ ]:
space = fit_coverage(study,variance_target=VARIANCE_TARGET)
print('Retained PCs:',space['n_pcs'],'training variance:',space['variance_retained'])
display(pd.DataFrame({'PC':np.arange(1,len(space['eigenvalues'])+1),
    'cumulative_training_variance':np.cumsum(space['eigenvalues'])/space['eigenvalues'].sum()}))


## Full training reference and training-only calibration

Project every training row. Merge exact duplicates **in float32 PCA coordinates** into unique locations before building an exact KDTree. This retains every represented training location, but removes repeated copies from neighbor counts; it measures spatial support rather than raw duplicate frequency. Projection/rounding collisions are not proof of identical raw records.

Coverage distance is the **mean Euclidean distance to five unique training locations**. For calibration, randomly select 3,000 training rows (so common source locations retain population weight), find six nearest unique sites, and remove the query's own site. Use the mean of the other five distances.

Freeze the calibration's 90th and 99th percentiles:

- **supported:** distance ≤ source-training 90th percentile;
- **intermediate:** between source-training 90th and 99th percentiles;
- **low_support:** above the 99th percentile.

These are descriptive source-relative bands, not calibrated probabilities or proven out-of-distribution boundaries. Evaluation queries are not members of the fitting reference; a legitimately repeated evaluation input may coincide with a training location. Hash-match sensitivity is therefore also useful. The 2017 held-out distances provide an independent check of how these bands behave on source-like data.

The tree and matrices are checkpointed. Choose a different `COVERAGE_OUT` subdirectory when changing geometric/query settings. The frozen baseline models and predictions are reused; there is no need to train them again. Cached settings are checked rather than silently overwritten.

In [ ]:
reference = coverage_reference(study,space,calibration_rows=CALIBRATION_ROWS,k_neighbors=K_NEIGHBORS)
display(pd.Series(reference))
calibration = pd.read_parquet(COVERAGE_OUT/'coverage_calibration.parquet')


## Query sampling and reporting populations

Use 15,000 uniform random queries from each evaluation split for **population-level coverage and performance summaries**. Supplement labels with fewer than 300 population queries, drawing uniformly among their remaining rows up to that count (or all available examples). Keep stable keys and flags identifying each sampling purpose.

Use the union for within-label plots/tables; each label's selected rows remain a uniform sample conditional on its count. Never mix the label-supplement rows into overall prevalence/performance summaries. Small categories can still yield very uncertain estimates. Classifier predictions themselves already cover all cleaned evaluation rows in the baseline notebook.

In [ ]:
queries = select_coverage_queries(study,POPULATION_QUERIES_PER_SPLIT,MINIMUM_QUERIES_PER_LABEL)
coverage = assess_coverage(study,space,reference,queries)
display(coverage.groupby(['split','population_sample','label_supplement']).size().rename('rows'))
population = coverage[coverage.population_sample]
counts = pd.crosstab(population.split,population.coverage_band).reindex(columns=['supported','intermediate','low_support'],fill_value=0)
display(counts)
display(counts.div(counts.sum(axis=1),axis=0))
counts.to_csv(COVERAGE_OUT/'coverage_population_counts.csv')


In [ ]:
summary,per_label = coverage_reports(study,coverage,models=registry['models'])
display(summary)
display(per_label)
# Explicitly inspect unknown protocols and training-hash matches as complementary diagnostics.
for model in registry['models']:
    joined=pd.read_parquet(COVERAGE_OUT/f'{model}_coverage_predictions.parquet')
    display(joined[joined.population_sample].groupby(['split','coverage_band']).agg(
        rows=('row_key','size'),unknown_protocol_rate=('unknown_protocol','mean'),
        training_feature_hash_match_rate=('training_feature_hash_match','mean')))


In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(13,4),constrained_layout=True)
for split,g in population.groupby('split'):
    values=np.sort(g.coverage_distance)
    axes[0].plot(values,np.arange(1,len(values)+1)/len(values),label=split)
    axes[1].hist(g.coverage_distance,bins=60,density=True,histtype='step',label=split)
for ax in axes:
    ax.axvline(reference['q90'],color='gray',ls='--',label='training q90')
    ax.axvline(reference['q99'],color='black',ls=':',label='training q99')
    ax.set_xlabel('Mean distance to 5 unique training locations');ax.legend();ax.grid(alpha=.2)
axes[0].set_ylabel('Cumulative fraction');axes[1].set_ylabel('Density')
plt.show()

bands=['supported','intermediate','low_support']
for metric,title,low,high in [('attack_recall','Attack recall','recall_ci_low','recall_ci_high'),
                              ('benign_fpr','Benign false-positive rate','fpr_ci_low','fpr_ci_high')]:
    fig,axes=plt.subplots(1,len(registry['models']),figsize=(13,4),squeeze=False,constrained_layout=True)
    for ax,model in zip(axes[0],registry['models']):
        for offset,(split,color) in zip([-.07,.07],[('test_2017','tab:blue'),('test_2018','tab:orange')]):
            g=summary.query('model == @model and split == @split and scope == "all"').set_index('coverage_band').reindex(bands)
            ax.errorbar(np.arange(3)+offset,g[metric],yerr=np.vstack([g[metric]-g[low],g[high]-g[metric]]),
                        fmt='o-',capsize=3,color=color,label=split)
        ax.set_xticks(range(3),bands);ax.set_ylim(-.02,1.02);ax.set_title(model);ax.set_ylabel(title);ax.legend();ax.grid(alpha=.2)
    plt.show()

fig,axes=plt.subplots(1,2,figsize=(14,5),sharex=True,sharey=True,constrained_layout=True)
colors={'supported':'tab:blue','intermediate':'tab:orange','low_support':'tab:red'}
for ax,(split,g) in zip(axes,sorted(population.groupby('split'))):
    for band in bands:
        subset=g[g.coverage_band==band]
        ax.scatter(subset.PC1,subset.PC2,s=3,alpha=.35,color=colors[band],label=band,rasterized=True)
    ax.set_title(split);ax.set_xlabel('PC1');ax.set_ylabel('PC2');ax.legend(markerscale=3);ax.grid(alpha=.2)
fig.suptitle('Training-fitted PCA: colors use distances in ALL retained PCs')
plt.show()


## Reading this recorded run

The following summary is generated from the saved tables and updates when those tables are regenerated.


In [ ]:
recorded_counts=pd.read_csv(COVERAGE_OUT/'coverage_population_counts.csv',index_col=0)
recorded_shares=recorded_counts.div(recorded_counts.sum(axis=1),axis=0)
for split,row in recorded_shares.iterrows():
    print(f"{split}: {row['supported']:.1%} supported; {row['intermediate']:.1%} intermediate; {row['low_support']:.1%} low support")
recorded_performance=pd.read_csv(COVERAGE_OUT/'coverage_performance.csv')
for model in registry['models']:
    g=recorded_performance.query('model == @model and split == "test_2018" and scope == "all"').set_index('coverage_band')
    print(f"{model}, 2018 attack recall: "+'; '.join(f"{band}: {g.loc[band,'attack_recall']:.1%} (n={int(g.loc[band,'attack_rows'])})" for band in ['supported','intermediate','low_support']))
small=recorded_performance.query('scope == "all" and attack_rows < 30')[['model','split','coverage_band','attack_rows','attack_recall','recall_ci_low','recall_ci_high']]
print('Small attack denominators: interpret intervals and avoid ranking models from these point estimates.')
display(small)
print('Differences between bands also reflect attack composition. Inspect the per-label tables before attributing the gap to geometric support.')


## Reading the results

First compare **full-evaluation baseline** performance with the sampled coverage table; do not substitute the sampled table for the baseline. Then ask whether 2018 has more low-support queries, whether errors concentrate there, and whether the relationship holds within individual attacks.

An overall coverage/error association can be driven by changing attack prevalence. Use `coverage_per_label.csv` to compare the same raw label across bands/years, without assuming that matching label names guarantee identical attacks. Low-support regions containing only benign or only attack examples cannot support every binary metric; counts and missing metrics are intentional.

The tables include pointwise Wilson intervals for recall/FPR, not capture-level uncertainty or multiple-comparison-adjusted inference. Review reference size, chosen PC count, calibration sampling, k, threshold percentiles, and hash matches before treating patterns as robust. Correlation between coverage and error does not establish causality; well-supported mistakes can reflect decision boundaries, labeling differences, or omitted features.

No classifier is retrained, no target-year threshold is selected, and no clustering is required. Clustering can later investigate coherent error pockets identified here.
`coverage_performance.csv` includes both the population sample as a whole and a sensitivity view excluding training feature-hash matches. The main plots use `scope == "all"`; compare the sensitivity rows before attributing supported-region success to generalization.
